<div align="center">

<img src="../images/Logo-Uni-Osnabrueck.jpg" width="300"/>

# Introduction to Computational Linguistics

</div>

## Minimum Edit Distance

### Why do we care?

How **similar** are two strings? This question shows up everywhere:

| Application | Example |
|-------------|---------|
| Spell correction | Typed `graffe` — did they mean `graf`, `graft`, `grail`, or `giraffe`? |
| Machine translation eval | Compare a system's output to a human reference |
| Speech recognition | Score the hypothesis against the reference transcript |
| Computational biology | Align two DNA sequences |
| Entity matching | `IBM Inc.` ≈ `IBM` |

### The three edit operations

The **minimum edit distance** is the smallest number of edits needed to turn one string into another. The allowed edits are:

| Operation | Example |
|-----------|---------|
| **Insert** a character | `cat` → `cats` |
| **Delete** a character | `cars` → `car` |
| **Substitute** one character for another | `cat` → `bat` |

### Example: `intention` → `execution`

Two strings of length 9, lined up against each other:

```
I N T E * N T I O N
E X E C U T I * O N
```

One possible alignment uses substitutions, one insertion (`*` in the first line), and one deletion (`*` in the second line).

### Two cost models

Different fields count edits differently:

| Model | Insert | Delete | Substitute |
|-------|--------|--------|------------|
| **Unit cost** | 1 | 1 | 1 |
| **Levenshtein** | 1 | 1 | **2** (= delete + insert) |

For `intention` ↔ `execution`:
- Unit-cost distance = **5**
- Levenshtein distance = **8**

We'll use Levenshtein in our code below — but you can switch by changing one number.

### Why we need Dynamic Programming

A naive search would try **every** possible sequence of edits. The number of paths grows exponentially with string length — even short words are hopeless.

**Key insight:** many of those paths end up at the same intermediate string. We don't need to remember all paths — just the **shortest cost** to reach each intermediate state. That's what dynamic programming does.

## Dynamic Programming

### The idea

Build a table `D` where `D[i][j]` is the edit distance between the first `i` characters of the source and the first `j` characters of the target.

The final answer is `D[n][m]`.

### Filling the table

Each cell looks at three neighbors and picks the cheapest:

| Neighbor | Operation | Added cost |
|----------|-----------|------------|
| `D[i-1][j]` (above) | delete a source char | `+1` |
| `D[i][j-1]` (left) | insert a target char | `+1` |
| `D[i-1][j-1]` (diagonal) | substitute / match | `+2` if different, `0` if same |

Base cases: `D[i][0] = i` and `D[0][j] = j`.

### Step 1 — Set up the table

We need an `(n+1) × (m+1)` grid of integers. The `+1` is for the row/column representing the **empty prefix**.

In [4]:
def edit_distance(source, target):
    """Levenshtein edit distance. Returns (distance, full DP table)."""
    n = len(source)
    m = len(target)

    # --- Step 1: build an (n+1) x (m+1) table filled with zeros ---
    D = []
    for i in range(n + 1):
        row = []
        for j in range(m + 1):
            row.append(0)
        D.append(row)

    # --- Step 2: fill in the base cases ---
    # First column: turning source[:i] into "" needs i deletions
    for i in range(n + 1):
        D[i][0] = i
    # First row: turning "" into target[:j] needs j insertions
    for j in range(m + 1):
        D[0][j] = j

    # --- Step 3: fill in the rest, one cell at a time ---
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Cost of arriving here by DELETING source[i-1]
            delete_cost = D[i-1][j] + 1

            # Cost of arriving here by INSERTING target[j-1]
            insert_cost = D[i][j-1] + 1

            # Cost of arriving here by SUBSTITUTION (or a free match)
            if source[i-1] == target[j-1]:
                substitute_cost = D[i-1][j-1]        # characters match, free
            else:
                substitute_cost = D[i-1][j-1] + 2    # Levenshtein sub cost = 2

            # Pick the cheapest of the three
            D[i][j] = min(delete_cost, insert_cost, substitute_cost)

    return D[n][m], D

### Step 2 — Visualize the table

It's much easier to *understand* the algorithm when you can see the matrix it builds. We'll wrap the DP table in a **pandas DataFrame** — Jupyter renders it as a clean HTML table with row and column labels.

In [5]:
import pandas as pd

def show_table(D, source, target):
    """Build a pandas DataFrame from the DP table.

    Rows are labelled with the source characters (plus '#' for the empty prefix),
    columns the same way for the target. Jupyter renders the result as an HTML table.
    """
    column_labels = ["#"] + list(target)
    row_labels    = ["#"] + list(source)
    return pd.DataFrame(D, index=row_labels, columns=column_labels)


# Try it on a tiny example first
distance, D = edit_distance("cat", "cats")
print("Edit distance:", distance)
show_table(D, "cat", "cats")

Edit distance: 1

       #   c   a   t   s
   #   0   1   2   3   4
   c   1   0   1   2   3
   a   2   1   0   1   2
   t   3   2   1   0   1


**How to read the table:**

- Each cell `D[i][j]` says: *the cheapest way to turn the first `i` chars of source into the first `j` chars of target costs this much*.
- Top-left corner `D[0][0] = 0` — turning `""` into `""` costs nothing.
- Bottom-right corner — the final answer for the full strings.

### Step 3 — Run it on the slide example

In [6]:
distance, D = edit_distance("intention", "execution")
print("Edit distance:", distance)
show_table(D, "intention", "execution")

Edit distance: 8

       #   e   x   e   c   u   t   i   o   n
   #   0   1   2   3   4   5   6   7   8   9
   i   1   2   3   4   5   6   7   6   7   8
   n   2   3   4   5   6   7   8   7   8   7
   t   3   4   5   6   7   8   7   8   9   8
   e   4   3   4   5   6   7   8   9  10   9
   n   5   4   5   6   7   8   9  10  11  10
   t   6   5   6   7   8   9   8   9  10  11
   i   7   6   7   8   9  10   9   8   9  10
   o   8   7   8   9  10  11  10   9   8   9
   n   9   8   9  10  11  12  11  10   9   8


## Backtrace — recovering the alignment

Edit distance gives us a **number**, but often we also want to know **which edits** produce it — the actual alignment.

Idea: every time we fill a cell, remember which neighbor we picked. Then, starting from the bottom-right corner, follow those pointers back to `(0, 0)`. The path you walk *is* the alignment.

| Pointer | Meaning | What it produces |
|---------|---------|------------------|
| `D` (diagonal) | came from `D[i-1][j-1]` | match or substitution |
| `L` (left) | came from `D[i][j-1]` | insertion (a char in target, gap in source) |
| `U` (up) | came from `D[i-1][j]` | deletion (a char in source, gap in target) |

In [7]:
def edit_distance_with_backtrace(source, target):
    n = len(source)
    m = len(target)

    # Cost table (same as before)
    D = []
    for i in range(n + 1):
        row = []
        for j in range(m + 1):
            row.append(0)
        D.append(row)

    # Pointer table: remembers which neighbor each cell came from
    P = []
    for i in range(n + 1):
        row = []
        for j in range(m + 1):
            row.append(None)
        P.append(row)

    # Base cases — these cells only have one valid predecessor
    for i in range(1, n + 1):
        D[i][0] = i
        P[i][0] = "U"    # only way to get here is from above (delete)
    for j in range(1, m + 1):
        D[0][j] = j
        P[0][j] = "L"    # only way to get here is from the left (insert)

    # Fill in the rest, recording both the cost and the pointer
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            delete_cost = D[i-1][j] + 1
            insert_cost = D[i][j-1] + 1
            if source[i-1] == target[j-1]:
                substitute_cost = D[i-1][j-1]
            else:
                substitute_cost = D[i-1][j-1] + 2

            best = min(delete_cost, insert_cost, substitute_cost)
            D[i][j] = best

            # Tie-breaking: prefer diagonal, then left, then up
            if best == substitute_cost:
                P[i][j] = "D"
            elif best == insert_cost:
                P[i][j] = "L"
            else:
                P[i][j] = "U"

    return D[n][m], D, P


def trace_back(P, source, target):
    """Walk from the bottom-right corner back to (0,0) following pointers."""
    aligned_source = ""
    aligned_target = ""
    operations = ""

    i = len(source)
    j = len(target)
    while i > 0 or j > 0:
        move = P[i][j]
        if move == "D":
            # Diagonal: match or substitution
            aligned_source = source[i-1] + aligned_source
            aligned_target = target[j-1] + aligned_target
            if source[i-1] == target[j-1]:
                operations = " " + operations    # match (no edit)
            else:
                operations = "s" + operations    # substitution
            i -= 1
            j -= 1
        elif move == "L":
            # Left: a character was inserted into the target
            aligned_source = "*" + aligned_source
            aligned_target = target[j-1] + aligned_target
            operations = "i" + operations
            j -= 1
        else:    # "U"
            # Up: a character was deleted from the source
            aligned_source = source[i-1] + aligned_source
            aligned_target = "*" + aligned_target
            operations = "d" + operations
            i -= 1

    return aligned_source, aligned_target, operations


# --- Run on the slide example ---
distance, D, P = edit_distance_with_backtrace("intention", "execution")
aligned_s, aligned_t, ops = trace_back(P, "intention", "execution")

print("Edit distance:", distance)
print()
print("Source:    ", "  ".join(aligned_s))
print("Target:    ", "  ".join(aligned_t))
print("Operations:", "  ".join(ops))
print()
print("Legend:  s = substitute,  i = insert,  d = delete,  ' ' = match,  * = gap")

Edit distance: 8

Source:     i  n  t  e  *  n  t  i  o  n
Target:     *  e  x  e  c  u  t  i  o  n
Operations: d  s  s     i  s            

Legend:  s = substitute,  i = insert,  d = delete,  ' ' = match,  * = gap


### Performance

| Resource | Cost |
|----------|------|
| Time | `O(n · m)` — we fill every cell once |
| Space (table) | `O(n · m)` |
| Backtrace | `O(n + m)` — at most one step per character |

Notice we only ever need the previous row, so memory can be reduced to `O(min(n, m))` — but then we lose the ability to backtrace.

## Weighted Edit Distance

Real-world spell-checkers don't treat every edit as equally likely. On a QWERTY keyboard, typing `e` instead of `r` is much more common than typing `e` instead of `m`. To capture this, we let each operation have its **own cost**:

- `del[x]` — cost of deleting character `x`
- `ins[y]` — cost of inserting character `y`
- `sub[x, y]` — cost of substituting `x` for `y`

The algorithm is exactly the same — we just replace the `+1` / `+2` constants with these functions.

In [8]:
def weighted_edit_distance(source, target, ins_cost, del_cost, sub_cost):
    """
    ins_cost(b)    -> cost of inserting character b
    del_cost(a)    -> cost of deleting character a
    sub_cost(a, b) -> cost of substituting a with b (should be 0 if a == b)
    """
    n = len(source)
    m = len(target)

    D = []
    for i in range(n + 1):
        row = []
        for j in range(m + 1):
            row.append(0)
        D.append(row)

    # Base cases: deletions / insertions from the empty prefix
    for i in range(1, n + 1):
        D[i][0] = D[i-1][0] + del_cost(source[i-1])
    for j in range(1, m + 1):
        D[0][j] = D[0][j-1] + ins_cost(target[j-1])

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost_del = D[i-1][j]   + del_cost(source[i-1])
            cost_ins = D[i][j-1]   + ins_cost(target[j-1])
            cost_sub = D[i-1][j-1] + sub_cost(source[i-1], target[j-1])
            D[i][j] = min(cost_del, cost_ins, cost_sub)

    return D[n][m], D


# --- Example: vowel-for-vowel typos are cheap, everything else is normal ---
vowels = "aeiou"

def my_sub(a, b):
    if a == b:
        return 0                    # no change
    if a in vowels and b in vowels:
        return 1                    # vowel-for-vowel: easy typo
    return 2                        # any other substitution

def my_ins(b):
    return 1

def my_del(a):
    return 1

for s, t in [("cat", "cot"), ("cat", "bat"), ("intention", "execution")]:
    d, _ = weighted_edit_distance(s, t, my_ins, my_del, my_sub)
    print(s, "→", t, " weighted distance =", d)

cat → cot  weighted distance = 1
cat → bat  weighted distance = 2
intention → execution  weighted distance = 7
